# Check-set selection (rebuild, 9 Oct 2026)

Builds two paper lists from `data/polymer_samples.db` (opened read-only). Nothing is written to disk; the results are the dataframes `check_df`, `needs_df`, `check_uids` and `needs_uids`:

- **Check set (60 papers):** 10 per source, 2–40 samples, every paper reports Tg, Td or tensile strength. Targets: Tg/Td/tensile ≥ 12 papers each, melt viscosity and solubility ≥ 5, every other property ≥ 3, and ≥ 15 *MW-complete* papers (≥ 2 of Mn/Mw/Đ **and** DP).
- **Needs-check set:** the 20 zero-sample papers (source quotas), plus the 4 papers with ≥ 200 samples and the 4 failed extractions.

The final cell gives `paper_uid` + `source` (+ cluster folder) so the full texts can be pulled from Arrhenius. Seeding is robust: `ORDER BY paper_uid` and separate RNGs, so the UIDs reproduce on any copy of the database.

In [1]:
# Cell 1: setup
import sqlite3, json, random, collections, os
from pathlib import Path
import pandas as pd

# find the repo root (folder containing data/polymer_samples.db) from wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/polymer_samples.db").exists())
conn = sqlite3.connect(f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True)

MVP = {"glass transition temperature", "thermal decomposition temperature", "tensile strength"}
ALL_PROPS = [
    "number-average molecular weight", "weight-average molecular weight",
    "dispersity", "degree of polymerization",
    "glass transition temperature", "melting temperature",
    "thermal decomposition temperature",
    "young's modulus", "tensile strength", "elongation at break",
    "storage modulus", "melt viscosity",
    "degree of crystallinity", "d-spacing", "particle size",
    "density", "solubility",
]
MW_TRIO = {"number-average molecular weight", "weight-average molecular weight", "dispersity"}
DP = "degree of polymerization"

PER_SOURCE = 10
MIN_S, MAX_S = 2, 40
TARGET = {p: 3 for p in ALL_PROPS}          # 2 required + 1 margin for classifier false positives
TARGET.update({p: 12 for p in MVP})         # 10 required + margin
TARGET.update({"melt viscosity": 5, "solubility": 5})
MWC_TARGET = 15                             # papers with >=2 of Mn/Mw/Đ and DP
MWC_BONUS = 3                               # gain bonus for an MW-complete paper until MWC_TARGET is reached

rng_pool = random.Random(7)                 # shuffles the check-set pool (tie-break order)
rng_zero = random.Random(11)                # draws the zero-sample papers


In [2]:
# Cell 2: candidate pool
rows = conn.execute("""
    SELECT p.paper_uid, p.canonical_source, p.category_class, p.fulltext_path,
           pc.properties_json,
           (SELECT COUNT(*) FROM samples s WHERE s.paper_uid = p.paper_uid) AS n
    FROM papers p
    JOIN property_classification pc USING (paper_uid)
    WHERE pc.has_property = 1
    ORDER BY p.paper_uid
""").fetchall()

pool = collections.defaultdict(list)
for uid, src, cat, path, pj, n in rows:
    if not (MIN_S <= n <= MAX_S):
        continue
    props = {x.strip().lower() for x in json.loads(pj)}
    if not props & MVP:
        continue
    mwc = len(props & MW_TRIO) >= 2 and DP in props
    pool[src].append(dict(uid=uid, src=src, cat=cat, path=path, props=props, n=n, mwc=mwc))

for s in sorted(pool):
    rng_pool.shuffle(pool[s])

pd.DataFrame({s: {"pool": len(c), "mw_complete": sum(r["mwc"] for r in c)} for s, c in sorted(pool.items())}).T


,pool,mw_complete
arxiv,106,2
elsevier,56773,851
europepmc,13253,343
s2orc,10515,151
springer_nature,170,2
wiley,29929,734


In [3]:
# Cell 3: greedy selection
cnt = collections.Counter()
per_src = collections.Counter()
used, chosen = set(), []
n_mwc = 0

def gain(r):
    g = sum(cnt[p] < TARGET[p] for p in r["props"])
    if r["mwc"] and n_mwc < MWC_TARGET:
        g += MWC_BONUS
    return g

while len(chosen) < PER_SOURCE * len(pool):
    best = None
    for s in sorted(pool):
        if per_src[s] >= PER_SOURCE:
            continue
        for r in pool[s]:
            if r["uid"] in used:
                continue
            g = gain(r)
            if best is None or g > best[0]:   # strict >: ties keep the shuffled order
                best = (g, r)
    r = best[1]
    chosen.append(r); used.add(r["uid"])
    per_src[r["src"]] += 1
    cnt.update(r["props"])
    n_mwc += r["mwc"]

check_df = pd.DataFrame([
    {"paper_uid": r["uid"], "source": r["src"], "category": r["cat"], "n_samples": r["n"],
     "mw_complete": r["mwc"], "n_props": len(r["props"]),
     "properties": "; ".join(sorted(r["props"])), "fulltext_path": r["path"]}
    for r in chosen
]).sort_values(["source", "paper_uid"]).reset_index(drop=True)
check_df.head()


,paper_uid,source,category,n_samples,mw_complete,n_props,properties,fulltext_path
0,ID000661306,arxiv,polymer_synthesis,2,False,2,density; glass transition temperature,data/fulltext/arxiv/ID000661306/ID000661306.pdf
1,ID001097753,arxiv,polymer_modification_combination,22,False,4,dispersity; glass transition temperature; numb...,data/fulltext/arxiv/ID001097753/ID001097753.pdf
2,ID001098286,arxiv,polymer_composite_formulation,6,False,1,glass transition temperature,data/fulltext/arxiv/ID001098286/ID001098286.pdf
3,ID001098538,arxiv,polymer_composite_formulation,4,False,3,glass transition temperature; storage modulus;...,data/fulltext/arxiv/ID001098538/ID001098538.pdf
4,ID001098910,arxiv,polymer_composite_formulation,7,False,5,density; glass transition temperature; melting...,data/fulltext/arxiv/ID001098910/ID001098910.pdf


In [8]:
for _, row in check_df.iterrows():
    print(f'{row["paper_uid"]} ({row["n_samples"]}, {row["n_props"]}): {row["properties"]}')

ID000661306 (2, 2): density; glass transition temperature
ID001097753 (22, 4): dispersity; glass transition temperature; number-average molecular weight; weight-average molecular weight
ID001098286 (6, 1): glass transition temperature
ID001098538 (4, 3): glass transition temperature; storage modulus; young's modulus
ID001098910 (7, 5): density; glass transition temperature; melting temperature; storage modulus; thermal decomposition temperature
ID001099328 (5, 2): glass transition temperature; storage modulus
ID001100019 (8, 3): dispersity; glass transition temperature; number-average molecular weight
ID001100849 (5, 2): elongation at break; tensile strength
ID001100979 (3, 2): glass transition temperature; storage modulus
ID001101339 (3, 3): dispersity; glass transition temperature; number-average molecular weight
ID000270446 (4, 10): degree of polymerization; dispersity; elongation at break; glass transition temperature; number-average molecular weight; solubility; storage modulus; t

In [4]:
# Cell 4: check that every target is met
coverage = pd.DataFrame({"papers": {p: cnt[p] for p in ALL_PROPS},
                         "target": TARGET})
coverage["ok"] = coverage.papers >= coverage.target
print("MW-complete papers:", n_mwc, "(target", MWC_TARGET, ")")
print("per source:", dict(check_df.source.value_counts()))
print("per category:", dict(check_df.category.value_counts()))
print("samples per paper: median", check_df.n_samples.median(), "| total", check_df.n_samples.sum())
assert coverage.ok.all() and n_mwc >= MWC_TARGET and len(check_df) == 60 and check_df.paper_uid.is_unique
coverage


MW-complete papers: 15 (target 15 )
per source: {'arxiv': np.int64(10), 'elsevier': np.int64(10), 'europepmc': np.int64(10), 's2orc': np.int64(10), 'springer_nature': np.int64(10), 'wiley': np.int64(10)}
per category: {'polymer_composite_formulation': np.int64(33), 'polymer_synthesis': np.int64(16), 'polymer_modification_combination': np.int64(11)}
samples per paper: median 5.0 | total 419


,papers,target,ok
number-average molecular weight,20,3,True
weight-average molecular weight,17,3,True
dispersity,20,3,True
degree of polymerization,15,3,True
glass transition temperature,43,12,True
melting temperature,20,3,True
thermal decomposition temperature,35,12,True
young's modulus,20,3,True
tensile strength,30,12,True
elongation at break,24,3,True


In [ ]:
# Cell 5: needs-check set
# 5a) the 20 zero-sample papers (successful extraction, no samples), drawn with source quotas
zero_rows = conn.execute("""
    SELECT p.paper_uid, p.canonical_source, p.category_class, p.fulltext_path
    FROM papers p JOIN sample_extraction_progress e USING (paper_uid)
    WHERE e.status = 'success'
      AND NOT EXISTS (SELECT 1 FROM samples s WHERE s.paper_uid = p.paper_uid)
    ORDER BY p.paper_uid
""").fetchall()
by_src = collections.defaultdict(list)
for r in zero_rows:
    by_src[r[1]].append(r)
print("zero-sample papers per source:", {s: len(v) for s, v in sorted(by_src.items())})

QUOTA = {"elsevier": 10, "wiley": 4, "s2orc": 4, "europepmc": 1, "springer_nature": 1}
zero = pd.DataFrame(
    [r for s, k in QUOTA.items() for r in rng_zero.sample(by_src[s], min(k, len(by_src[s])))],
    columns=["paper_uid", "source", "category", "fulltext_path"],
).assign(n_samples=0, reason="0 samples")

# 5b) papers with >= 200 samples (possible over-splitting)
big = pd.read_sql("""
    SELECT p.paper_uid, p.canonical_source AS source, p.category_class AS category,
           COUNT(*) AS n_samples, p.fulltext_path
    FROM samples s JOIN papers p USING (paper_uid)
    GROUP BY p.paper_uid HAVING n_samples >= 200
    ORDER BY n_samples DESC, p.paper_uid
""", conn).assign(reason=">=200 samples")

# 5c) failed extractions
failed = pd.read_sql("""
    SELECT p.paper_uid, p.canonical_source AS source, p.category_class AS category,
           p.fulltext_path, e.last_error
    FROM papers p JOIN sample_extraction_progress e USING (paper_uid)
    WHERE e.status = 'failed'
    ORDER BY p.paper_uid
""", conn).assign(n_samples=0)
failed["reason"] = "extraction failed: " + failed.pop("last_error").fillna("").str[:60]

needs_df = pd.concat([zero, big, failed], ignore_index=True)[
    ["paper_uid", "source", "category", "n_samples", "reason", "fulltext_path"]]
print(needs_df.reason.str.split(":").str[0].value_counts().to_dict())
needs_df


zero-sample papers per source: {'elsevier': 1108, 'europepmc': 39, 's2orc': 92, 'springer_nature': 1, 'wiley': 93}
{'0 samples': 20, '>=200 samples': 4, 'extraction failed': 4}


,paper_uid,source,category,n_samples,reason,fulltext_path
0,ID000691671,elsevier,polymer_synthesis,0,0 samples,data/fulltext/elsevier/ID000691671/ID000691671...
1,ID000692031,elsevier,polymer_synthesis,0,0 samples,data/fulltext/elsevier/ID000692031/ID000692031...
2,ID000691661,elsevier,polymer_synthesis,0,0 samples,data/fulltext/elsevier/ID000691661/ID000691661...
3,ID000693070,elsevier,polymer_synthesis,0,0 samples,data/fulltext/elsevier/ID000693070/ID000693070...
4,ID000596706,elsevier,polymer_composite_formulation,0,0 samples,data/fulltext/elsevier/ID000596706/ID000596706...
5,ID000595965,elsevier,polymer_modification_combination,0,0 samples,data/fulltext/elsevier/ID000595965/ID000595965...
6,ID000694179,elsevier,polymer_modification_combination,0,0 samples,data/fulltext/elsevier/ID000694179/ID000694179...
7,ID000692254,elsevier,polymer_composite_formulation,0,0 samples,data/fulltext/elsevier/ID000692254/ID000692254...
8,ID000596351,elsevier,polymer_composite_formulation,0,0 samples,data/fulltext/elsevier/ID000596351/ID000596351...
9,ID000195898,elsevier,polymer_modification_combination,0,0 samples,data/fulltext/elsevier/ID000195898/ID000195898...


In [6]:
# Cell 6: final lists for downloading from Arrhenius (uid, source, paper folder on the cluster)
def to_uids(df):
    return df.assign(folder=df.fulltext_path.map(lambda p: str(Path(p).parent) + "/"))[
        ["paper_uid", "source", "folder"]].reset_index(drop=True)

check_uids = to_uids(check_df)
needs_uids = to_uids(needs_df).assign(reason=needs_df.reason.values)
print(len(check_uids), "check-set papers |", len(needs_uids), "needs-check papers")
display(check_uids)
display(needs_uids)


60 check-set papers | 28 needs-check papers


,paper_uid,source,folder
0,ID000661306,arxiv,data/fulltext/arxiv/ID000661306/
1,ID001097753,arxiv,data/fulltext/arxiv/ID001097753/
2,ID001098286,arxiv,data/fulltext/arxiv/ID001098286/
3,ID001098538,arxiv,data/fulltext/arxiv/ID001098538/
4,ID001098910,arxiv,data/fulltext/arxiv/ID001098910/
5,ID001099328,arxiv,data/fulltext/arxiv/ID001099328/
6,ID001100019,arxiv,data/fulltext/arxiv/ID001100019/
7,ID001100849,arxiv,data/fulltext/arxiv/ID001100849/
8,ID001100979,arxiv,data/fulltext/arxiv/ID001100979/
9,ID001101339,arxiv,data/fulltext/arxiv/ID001101339/


,paper_uid,source,folder,reason
0,ID000691671,elsevier,data/fulltext/elsevier/ID000691671/,0 samples
1,ID000692031,elsevier,data/fulltext/elsevier/ID000692031/,0 samples
2,ID000691661,elsevier,data/fulltext/elsevier/ID000691661/,0 samples
3,ID000693070,elsevier,data/fulltext/elsevier/ID000693070/,0 samples
4,ID000596706,elsevier,data/fulltext/elsevier/ID000596706/,0 samples
5,ID000595965,elsevier,data/fulltext/elsevier/ID000595965/,0 samples
6,ID000694179,elsevier,data/fulltext/elsevier/ID000694179/,0 samples
7,ID000692254,elsevier,data/fulltext/elsevier/ID000692254/,0 samples
8,ID000596351,elsevier,data/fulltext/elsevier/ID000596351/,0 samples
9,ID000195898,elsevier,data/fulltext/elsevier/ID000195898/,0 samples


## Transfer from Arrhenius

Copies every file in each paper folder (`<source>/<uid>/`: original, Markdown, …) with `rsync`, one call per source. The ssh connection is shared between calls, so you authenticate once.

- Check set (60) → `data/fulltext/<source>/<uid>/`
- Needs-check set → `data/inspect/sample_extraction/<uid>/`

Set `DRY_RUN = True` to see what would be copied first. Re-running only copies what is missing or changed.

In [ ]:
# # Transfer 1: check set (60 papers) -> data/fulltext/<source>/<uid>/
# import subprocess, tempfile

# REMOTE = "arrhenius:/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/data/fulltext"
# SSH = "ssh -o ControlMaster=auto -o ControlPath=~/.ssh/cm-%r@%h:%p -o ControlPersist=10m"
# DRY_RUN = False

# def rsync_from_arrhenius(df, dest_for_source):
#     """One rsync per source: copy REMOTE/<source>/<uid>/ into dest_for_source(source)/<uid>/."""
#     for src, uids in df.groupby("source").paper_uid:
#         dest = dest_for_source(src)
#         dest.mkdir(parents=True, exist_ok=True)
#         with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
#             f.write("".join(f"{u}/\n" for u in sorted(uids)))
#         cmd = ["rsync", "-a", "-r", "-v", "-e", SSH, f"--files-from={f.name}",
#                f"{REMOTE}/{src}/", f"{dest}/"]
#         if DRY_RUN:
#             cmd.insert(1, "--dry-run")
#         res = subprocess.run(cmd, capture_output=True, text=True)
#         print(f"--- {src}: {len(uids)} papers -> {dest.relative_to(ROOT)}  (exit {res.returncode})")
#         print(res.stdout[-1500:] if res.returncode == 0 else res.stderr)
#         os.unlink(f.name)

#     if not DRY_RUN:   # every uid folder should now exist locally and not be empty
#         missing = [u for s, u in zip(df.source, df.paper_uid)
#                    if not (dest_for_source(s) / u).is_dir() or not any((dest_for_source(s) / u).iterdir())]
#         print("missing or empty folders:", missing or "none")

# rsync_from_arrhenius(check_uids, lambda src: ROOT / "data/fulltext" / src)


In [12]:
# # Transfer 2: needs-check set -> data/inspect/sample_extraction/<uid>/  (flat, no source level)
# INSPECT = ROOT / "data/inspect/sample_extraction"
# rsync_from_arrhenius(needs_uids, lambda src: INSPECT)


## Zero-sample papers: why no samples were extracted (inspected 9 Oct 2026)

Read the Markdown and the original file (XML / PDF / JSON) for each of the 20 zero-sample papers in `data/inspect/sample_extraction/`. "Probable extractor miss" is judged by reading the paper; the extractor was not re-run.

| # | UID | Source | MD words | Verdict | What happened |
|---|---|---|---|---|---|
| 0 | ID000691671 | elsevier | 81 | Conversion failure | XML has no structured body; full text (1,263 words) only in `<rawtext>` (scanned, unstructured), which the converter drops, so the MD is title + abstract. PEO–pyromellitimide copolymers, *Synth. Met.* 1997. |
| 1 | ID000692031 | elsevier | 45 | Conversion failure | Same `<rawtext>`-only XML (917 words). Poly(alkylthiophene) copolymers synthesised, *Synth. Met.* 1997. |
| 2 | ID000691661 | elsevier | 132 | Conversion failure | Same (1,076 words). Cumulene-containing polymer, nonlinear optics, *Synth. Met.* 1997. |
| 3 | ID000693070 | elsevier | 108 | Conversion failure | Same (1,677 words). Alkoxy-PPP derivatives synthesised by Suzuki coupling, *Synth. Met.* 1999. |
| 4 | ID000596706 | elsevier | 163 | Conversion failure | Same (3,471 words). Particle-filled polymer composites, viscoelastic parameters, *Procedia Eng.* 2015. |
| 5 | ID000595965 | elsevier | 104 | Conversion failure | Same (4,110 words). PA12 laser-sintering powder, hydrolysis, *Procedia CIRP* 2020. |
| 6 | ID000694179 | elsevier | 97 | Conversion failure | Same (1,198 words). Polyurethanes with varied crosslink density, *Physica B* 1997. |
| 7 | ID000692254 | elsevier | 112 | Conversion failure | Same (2,214 words). PEDOT/CF₃SO₃ films, *Synth. Met.* 1998. |
| 8 | ID000596351 | elsevier | 158 | Conversion failure | Same (3,645 words). Fibre-reinforced geopolymer composites, *Procedia Eng.* 2011. |
| 9 | ID000195898 | elsevier | 122 | Conversion failure | Same (5,822 words). Radiation grafting with acid/salt additives, *Radiat. Phys. Chem.* 1990. |
| 10 | ID000079910 | wiley | 2,223 | Conversion failure | PDF→MD stops mid-sentence at a figure after ~32% of the paper (9 pages, ~7,000 words), before the polymerisation part. The polysarcosine polymers would be samples. |
| 11 | ID000050366 | wiley | 364 | Conversion failure | Only abstract + start of introduction (10% of 3,516 words). Supramolecular polymers of perylenes 1–3. |
| 12 | ID000035979 | wiley | 6,317 | Probable extractor miss | Fully converted. P4T2F-HD:Y6-BO and P3HT:Y6-BO blend films are made and characterised (TEM, GIWAXS, DSC); polymer synthesis is only in the SI, so the blends are the samples. |
| 13 | ID000030083 | wiley | 448 | Conversion failure | Only title, garbled abstract and affiliations (7% of 6,499 words). Elastomers with aligned perovskite nanosheets at several volume fractions would be samples. |
| 14 | ID000963830 | s2orc | 2,015 | Genuine (borderline) | Account of the authors' earlier NHC–Zr catalyst work; PLA results (Đ < 1.1, Table 1) are cited from their previous papers. Text also starts mid-sentence (common S2ORC gap). |
| 15 | ID001080532 | s2orc | 789 | Borderline / out of scope | Drilling-fluid muds with a local plum tree gum vs commercial additives. Formulations are made, but this is petroleum engineering, closer to a classifier false positive. |
| 16 | ID001052168 | s2orc | 1,838 | Probable extractor miss | Polysiloxane liquid-crystal polymers "No. 1–5" with Sn₂P₂S₆ nanoparticles in electro-optic cells. Poorly written, but samples are made and measured. |
| 17 | ID000234303 | s2orc | 4,039 | Probable extractor miss | Sulfonated polyacrylamide + surfactant + alcohol solutions prepared and tested for oil recovery (rheology, surface tension). Formulations from commercial polymers, arguably samples. |
| 18 | ID001041028 | europepmc | 6,245 | Genuine | Commercial PU shape-memory polymer (MM4520, SMP Technologies); specimens only cut from a sheet and annealed. No material created in the paper. |
| 19 | ID000726343 | springer_nature | 1,099 | Genuine | *News & Views* commentary on carbonized polymer dots (confirmed in the XML); no experimental work. |

**Summary:** 13 conversion failures (all 10 Elsevier, 3 of 4 Wiley), 3 probable extractor misses, 3 genuine, 1 borderline.

- **Elsevier:** the XML→MD converter ignores `<rawtext>` when there is no structured body. All 10 sampled papers (mostly 1990s scans and *Procedia* proceedings) have this pattern, so it likely explains most of the 1,108 Elsevier zero-sample papers. Fix: fall back to `<rawtext>` when the XML has no sections.
- **Wiley:** truncated PDF→MD conversions (MD has 7–32% of the `pdftotext` word count). A MD-vs-PDF word-count ratio would flag these across the corpus.
- **Extractor misses** are guideline questions rather than clear bugs: blend films whose polymer synthesis is in the SI, and formulations made from commercial polymers. Decide whether blends and solutions count as samples.


### Manually Checking a few papers

In [2]:
# Manual check: get the extracted samples for a list of paper UIDs
import sqlite3, json, textwrap
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/polymer_samples.db").exists())
_conn = sqlite3.connect(f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True)

def get_samples(uids):
    """One row per extracted sample for the given paper UIDs, with the paper's source and local Markdown path."""
    uids = list(dict.fromkeys(uids))                      # keep order, drop duplicates
    q = f"""
        SELECT s.paper_uid, p.canonical_source AS source, s.sample_id,
               s.name, s.identifier, s.aliases, s.description
        FROM samples s JOIN papers p USING (paper_uid)
        WHERE s.paper_uid IN ({",".join("?" * len(uids))})
        ORDER BY s.paper_uid, s.sample_id
    """
    df = pd.read_sql(q, _conn, params=uids)
    df["aliases"] = df.aliases.map(lambda a: json.loads(a) if a else [])
    df["paper_uid"] = pd.Categorical(df.paper_uid, categories=uids, ordered=True)   # keep input order
    df = df.sort_values(["paper_uid", "sample_id"]).reset_index(drop=True)
    df["paper_uid"] = df.paper_uid.astype(str)
    df["md_path"] = [next((str(f.relative_to(ROOT)) for f in (ROOT / "data/fulltext" / s / u).glob("*.md")), None)
                     for s, u in zip(df.source, df.paper_uid)]
    missing = [u for u in uids if u not in set(df.paper_uid)]
    if missing:
        print("no samples in the db for:", missing)
    return df

def show_samples(df, width=110):
    """Readable printout for manual checking: one block per paper, one numbered entry per sample."""
    for uid, g in df.groupby("paper_uid", sort=False):
        print("=" * width)
        print(f"{uid}  ({g.source.iloc[0]})  {len(g)} samples")
        print(f"paper: {g.md_path.iloc[0] or 'not downloaded'}")
        print("=" * width)
        for i, r in enumerate(g.itertuples(), 1):
            print(f"[{i}] {r.name}   | id: {r.identifier or '-'}   | aliases: {', '.join(r.aliases) or '-'}   (sample_id {r.sample_id})")
            print(textwrap.fill(r.description, width, initial_indent="    ", subsequent_indent="    "))
            print()


In [3]:
# Paste the UIDs to check here
uids = [
    "ID000270446",
]

manual_df = get_samples(uids)
print(manual_df.groupby("paper_uid", sort=False).size().to_string(), "\n")
show_samples(manual_df)


paper_uid
ID000270446    4 

ID000270446  (elsevier)  4 samples
paper: data/fulltext/elsevier/ID000270446/ID000270446.md
[1] Alginic acid acetate   | id: AlgAc   | aliases: AlgAc   (sample_id 278549)
    An ester derivative of alginic acid prepared by fully esterifying the hydroxyl groups of alginic acid with
    acetic acid and acetic anhydride. Alginic acid was pretreated by swelling in 200 mL distilled water for 1
    h, exchanging with acetone, and stirring in 100 mL acetic acid before filtration. The pretreated material
    was reacted in 50 mL acetic anhydride at 45 °C with a mixture of 50 mL acetic acid and 0.5 mL 60% aqueous
    perchloric acid catalyst for 3 h. The reaction mixture was poured into 1 L distilled water, and the
    precipitate was filtered, washed with distilled water, and dried in vacuo overnight. The degree of
    substitution (DS) is 2.0.

[2] Alginic acid propionate   | id: AlgPr   | aliases: AlgPr   (sample_id 278550)
    An ester derivative of alginic acid

## Sample extraction evaluation: precision only (v2, 13 Oct 2026)

**What was judged.** All **419 extracted sample records** from the **60 check-set papers** (10 per source), read against each paper's full text. Only precision: for every record that *was* extracted, is it a real sample, is its name/identifier/description factually right, and does it point to exactly one sample? Missing samples were not looked for.

**Two independent judges**, same prompt (precision-only judge, v2):
- **Claude:** one subagent per source → `claude_workspace/sample_check_v2/{source}_precision.csv` + `_precision_report.md`
- **Codex:** → `codex_workspace/sample_check_v2/{source}_precision.csv` + `_precision_report.md`

Each record gets three yes/no answers (`is_sample`, `description_correct`, `identifiable`) and one verdict: `correct`, `not_a_sample`, `incorrect` (wrong fact) or `ambiguous` (maps to more than one sample).

**Sample definition used by the judges:** a final material prepared in the paper (synthesised, formulated, blended, compounded or chemically modified). Not samples: intermediates, test specimens, commercial materials used as received or only shaped, small molecules/fillers on their own, inorganic-only materials, materials cited from other work.

### Headline numbers

| Source | Records | Claude precision | Codex precision | Agreement on "correct?" |
|---|---|---|---|---|
| s2orc | 39 | 84.6% | 84.6% | 100% |
| wiley | 60 | 76.7% | 73.3% | 97% |
| europepmc | 79 | 72.2% | 68.4% | 94% |
| arxiv | 65 | 69.2% | 64.6% | 89% |
| springer_nature | 92 | 79.3% | 60.9% | 75% |
| elsevier | 84 | 82.1% | 52.4% | 70% |
| **Total** | **419** | **77.1%** (323) | **65.2%** (273) | **85%** (κ = 0.65) |

| Verdict | Claude | Codex |
|---|---|---|
| correct | 323 | 273 |
| not_a_sample | 51 | 81 |
| ambiguous | 23 | 44 |
| incorrect | 22 | 21 |

Agreement per question: `is_sample` 91%, `description_correct` 94%, `identifiable` 93%. Full verdict agreement 83% (κ = 0.64). For comparison, the v1 judge (no sample definition) gave 77% agreement, κ = 0.59.

**Split of the 419 records:** both judges say correct **267** · both flag an error **90** · disagree **62**.

### 1. Where both judges agree the extraction is right (267 records)

Typical correct records: each member of a composition or loading series as its own record with the paper's code, and a description that says what differs.

| Paper | What the extractor extracted | Claude | Codex |
|---|---|---|---|
| ID000977754 s2orc | PEO, PEO:pLLZO-10/-20, PEO:nwLLZO-10/-20 (5 membranes) | correct 5/5 | correct 5/5 |
| ID000186664 europepmc | Mb-Ch, Mb-Ch-Ex.Av, Mb-Ch-Ex.Av-NPs (chitosan membranes) | correct 3/3 | correct 3/3 |
| ID000572215 elsevier | RGE…RGDd cured ricinoleic-acid polyesters (8) | correct 8/8 | correct 8/8 |
| ID000180958 s2orc | Silica anion exchanger, 1-, 2-, 3-, 4-, 7-, 11-, 15-layer (7) | correct 7/7 | correct 7/7 |
| ID001100019 arxiv | c-PMMA129…279-b-FMA (4 cyclic) | correct 4/4 | correct 4/4 |

Under the v2 definition, cases flagged in the earlier manual check are no longer errors. RGE…RGDd: the prepolymer is an intermediate, so one record per cured network is right. Codes shared with test specimens (hot-pressed bars, spin-coated films) are not ambiguity.

### 2. Where both judges flag an error (90 records)

| Error type | Records | Paper | What the extractor extracted | Claude | Codex |
|---|---|---|---|---|---|
| **Commercial polymer only shaped** | 6 + 2 | ID001097753, ID001098286 arxiv | Monodisperse PS from Polymer Source, spin-coated films; neat PS50k/PS100k matrix films | not_a_sample: supplied PS only spin-coated | not_a_sample: commercial, only spin-coated |
| | 3 | ID000742866 europepmc | Plain-, CF- and ESD-PEEK parts | not_a_sample: commercial filament/stock only printed or machined | not_a_sample |
| | 5 | ID001037224, ID000061162, ID000067020, ID000593925, ID000415115 | Neat PCL, neat PLA, neat PP "AP0", maize starch film, neat PCL nanofibers | not_a_sample: reference polymer only melted/pressed/cast | not_a_sample |
| **Out of scope** | 9 | ID000189836 springer_nature | NG0–NG5, NSF0–NSF7 geopolymer mixes | not_a_sample: inorganic | not_a_sample: inorganic |
| **Intermediate** | 4 + 1 | ID000317612 elsevier | RAFT-PDMS10–40 macro-RAFT agents; PA-RAFT0-8 emulsion | not_a_sample: precursor of the emulsifier; PA-RAFT0-8 incorrect (recipe never given in paper) | not_a_sample |
| | 10 | ID000720027 europepmc | Acetal-protected P1–P6 / P1c (later deprotected) | not_a_sample; 3 incorrect (50 °C copied from entry 1, footnote says 25 °C/70 h) | not_a_sample |
| | 2 + 1 + 1 + 1 + 1 | ID001002844, ID000180835, ID000415115, ID000573272, ID000085482 | Compatibilizer precursors C1/C2; AN/AMPS before amidoximation; EtOH lignin fraction; BHET oligomer feedstock; dry OEGMA/BnMA matrix film | not_a_sample | not_a_sample |
| **Not in the paper** | 3 + 3 | ID001044877 europepmc; ID000061157 wiley | PDMS-b-PEO-only filter papers at 0.5/1.5/2.0%; unirradiated B3–B5 films | not_a_sample: no results reported | not_a_sample / incorrect |
| **Series merged into one record** | 6 | ID000137005 wiley | UHMWPE/CB-20, /MWCNTs-3, /GNP-10 … with no orientation level | ambiguous: non-oriented + several draw levels | ambiguous |
| | 3 + 2 | ID000799604, ID000797992 s2orc | PVA, PVA-G, PVA-G-HA fibres; SCFNR-1/-10/APA6 | ambiguous: as-spun + 5 heat treatments; 4 fibre vol% | ambiguous |
| | 3 | ID000414493 elsevier | PLLA/ScLNPs-g-PDLA composites | ambiguous: 0.5–15 wt% merged | ambiguous |
| | 7 | ID000732356 springer_nature | poly1, poly2, poly3 (ROP) | ambiguous/incorrect: Table 2 entries merged; ranges "1–6 mol%", "1–4 days" wrong | ambiguous/incorrect |
| | 2 | ID000997875 europepmc | ASPA-pAMPS microgels, unloaded + drug-loaded | ambiguous/incorrect: AFn-1…9 merged; "chemically modified" not in paper | same |
| | 3 | ID000573657 elsevier | CAC/PCL1250 etc. | ambiguous: photocure time missing | not_a_sample: uncured resin |
| **Wrong numbers from a garbled table** | 10 | ID001097753 arxiv | PS blend records (S2fw, S5bi …) | incorrect: Mw 44.2 for 442, 20% for 22%, missing component; Markdown Table I lost digits | incorrect |
| **Inferred condition** | 2 | ID000061157 wiley | Unirradiated B1/B2 films "annealed at 100 °C" | incorrect (B1 also commercial PVA) | incorrect: annealing only after irradiation |

**Notes for the next round.** Under the refined intermediate rule below, two groups are worth re-checking: the protected P1 series (ID000720027, Tg is reported per DP) and the dry matrix film (ID000085482, Tg/Td reported). Both may count as samples.

### 3. Where the judges disagree (62 records)

In every case the extracted record is the same. The judges read the rule differently, or one judge caught a fact the other missed.

**(a) Is a material that is later converted a sample? (26 records).** Codex: anything later converted is an intermediate. Claude: it is a sample if the paper characterises it as a material in its own right.

| Paper | What the extractor extracted | Later used to make | Claude | Codex |
|---|---|---|---|---|
| ID001100019 arxiv | l-PMMA129…279-b-FMA1 (4) | cyclic PMMA | correct: Tg profiles measured and compared | not_a_sample |
| ID000317612 elsevier | PA-RAFTn-Y emulsions (8) | isocyanate-cured coatings | correct | not_a_sample: uncured resin |
| ID000414493 elsevier | ScLNPs (3), ScLNPs-g-PDLA (3) | PLLA composites | correct | not_a_sample: filler |
| ID000573272 elsevier | PET-EG0.03…ZA0.5% BHET oligomers (4) | r-PET, PETMG | correct | not_a_sample |
| ID000720027 europepmc | P1 Table 1 entry | P1d | correct | not_a_sample |
| ID000732392 springer_nature | starch acetoacetate | PLA/SAA blends | correct | not_a_sample |
| ID000931272 springer_nature | GA-g-Ch | PU formulations | correct | not_a_sample |
| ID000143254 wiley | waterborne acrylic resin | water-based inks | correct | not_a_sample |

**Decision (13 Oct): Claude's reading.** A material that is later converted is still a sample if it is characterised separately (e.g. linear PMMA).

**(b) Ageing / durability exposure (15 records).** The extractor made one record per formulation, never separate aged versions.

| Paper | What the extractor extracted | Paper also does | Claude | Codex |
|---|---|---|---|---|
| ID000905310 springer_nature | PLA/PBS/PP/PE-Jute, 10 and 30 wt% (8) | hydrothermal ageing | correct | ambiguous: no ageing state |
| ID000573272 elsevier | PETMG30, PETMG40 | 360 h thermo-oxidative / hygrothermal ageing | correct | ambiguous |
| ID001037224 europepmc | PCL amine-/HO-/Gly-POSS (3) | ageing at 100 °C, shearing | correct | ambiguous |
| ID000415115 elsevier | PCL/KL5, PCL/ET5 nanofibers | 72 h UV exposure | correct | ambiguous |

**Proposed: Claude's reading.** Ageing, weathering and UV/hydrothermal exposure are measurement conditions, not new samples.

**(c) Factual error caught by only one judge (11 records).** These are taken as **incorrect**: the union of both judges.

| Paper | What the extractor wrote | Caught by | Problem |
|---|---|---|---|
| ID000185901 springer_nature | EPDM/kaolinite 1/3/5/7 phr: "solvent was extracted by dipping the mould into water" | Codex | paper: mould dipped in water *after* solvent extraction |
| ID000732356 springer_nature | "poly1-b-PEG", "poly2-b-PEG" | Codex | identifier not used in the paper |
| ID000661306 arxiv | polyimide P130093: "7.5 wt.%" casting solution | Codex | paper says 7.5% (no wt basis) |
| ID000661306 arxiv | polyimide P432352: "1.6 wt.%" | Claude | paper says 1.6% w/v |
| ID000752848 europepmc | Homo PBS with [COOH]/[OH] = 1.2 | Claude | ratio stated only for the CNC mixtures |
| ID000732493 springer_nature | H5.42 blend "highest fracture toughness"; pure PU prepared like the blends | Claude | claim not in paper; procedure copied |

**(d) Still open (10 records).**

| Paper | What the extractor extracted | Claude | Codex |
|---|---|---|---|
| ID000732356 springer_nature | poly1/2/3 for Table 4 entries 4–6, Table 2 entry 10 (4) | correct | ambiguous: one entry sampled at several reaction times |
| ID000414493 elsevier | PLLA/ScLNPs composite (3) | correct: only 10 wt% reported | ambiguous: several loadings |
| ID000150714 wiley | PProP | correct | ambiguous: pressed bulk vs electrospun fibre compared |
| ID001099328 arxiv | PANI-GNP nanocomposite | not_a_sample: filler intermediate | correct |
| ID000726469 springer_nature | PU mat (commercial Elastollan, electrospun) | not_a_sample: commercial, only shaped | correct |

### 4. Adjudicated result and next steps

Applying (a) and (b) as decided/proposed, taking the union of factual errors in (c), and counting the commercial electrospun PU mat as not a sample:

| | Records |
|---|---|
| correct | **308** (267 agreed + 26 intermediate rule + 15 ageing rule) |
| error | **102** (90 agreed + 11 factual + 1 PU mat) |
| open | 9 |
| **Precision** | **73.5%**, up to 75.7% if all open records are correct |

**What the extractor gets wrong, in order:**
1. **Extracting materials that are not samples** (~50): commercial references that were only shaped (PS, PEEK, neat PCL/PLA/PP, starch), intermediates such as precursors and macro-RAFT agents, an out-of-scope geopolymer paper, and controls with no results.
2. **Merging a series into one record** (~25): loadings, heat-treatment or orientation levels, several table entries. The description then gives ranges.
3. **Wrong facts** (~25): 10 come from one garbled Markdown table. The rest are conditions copied from another entry, unit/basis slips (w/v vs wt%), invented identifiers, and result claims.

**Rule decisions to carry into the judge and the extraction prompt:**
- **Intermediate:** a material made only as a step towards another sample and characterised only to confirm that step (e.g. a prepolymer checked by GPC, then cured). A material that is later converted or used as a component is still a sample if the paper characterises it separately as a material in its own right (e.g. linear PMMA, starch acetoacetate).
- **Ageing, weathering, degradation, UV/hydrothermal exposure:** measurement conditions, not samples. Treatment levels are processing steps that make the material (curing, annealing, drawing, crosslinking irradiation).
- **Test specimens and commercial polymers used as received or only shaped:** not samples.

**Extraction prompt fixes these point to:** exclude commercial materials used as references (the current prompt excludes them only as constituents); one record per series member / table entry / treatment level, each with its value; no ranges; never copy conditions from another entry; keep units as written; no results in descriptions.